# Senate Stage 1: lagged Split Ticket WAR (feature ablation)

Does the nominees' past Senate **Wins Above Replacement** (Split Ticket) improve S1, and does any gain survive the
Stage 2 blend with polls and expert ratings? An experiment: production S1, the blend weights and the forecasts are
unchanged.

**The predictor** (`src/congress/war.py`; data from `data-processing/split_ticket_war.py`):
- **WAR:** Split Ticket's per-race residual against a baseline that controls for incumbency, lagged presidential
  partisanship and swing, spending and demographics (the Senate baseline is fit on the cycle's House races). It is
  D positive and centered by chamber-year (Senate WAR averages about D+2 every cycle, section 1).
- **Whose races:** earlier **Senate** WAR races in which either current nominee was the D or R nominee, matched by FEC
  candidate ID. A senator's House races don't count (different baseline, different FEC ID).
- **The lag:** one earlier year, that value; two or more, $0.75\,w_{recent} + 0.25\,w_{previous}$. Never the race's
  own year.
- **Missing WAR is neutral**, exactly as in `02_reg_lagged_performance.ipynb`'s `S1 + L (neutral)`
  (`lagged.fit_predict_lagged(..., missing_lag="neutral")`): a race with no earlier WAR gets `war_lag = 0`, flagged,
  so **no training or scored row is dropped**. Both models train on S1's rows (asserted in section 2). No separate
  missing-WAR coefficient.

**Specification:** S1 (`partisan_lean + gen_margin_raw`, `inc_signed` fixed at +2.5) plus `war_lag`, one persistence
coefficient, equal-year WLS. **Blend:** `stage2.blend` with the production weights (0.37 / 0.54 / 0.09 with polls,
0.65 / 0 / 0.35 without), unchanged.

**What limits this test:**
- **Coverage.** WAR starts in 2014. Senators re-run every six years, so 2018's races (class I, last elected in 2012)
  have **no** lags, and the 2018 and 2020 windows learn the coefficient from the **3** 2016 races that have one. Only
  the 2022 window (31 training races with a lag) and the 2024 window (56) identify it. Section 3 reports all four
  test years as specified, and 2022–2024 separately.
- **Vintage.** One WAR vintage (refreshed May 2026, 2018–2022 recomputed in January 2025 with the 2024 method), so
  past WAR was produced with hindsight on the method, and most of it didn't exist when these elections were
  forecast. See `data-processing/README.md`.
- **Small samples:** 134 scored races, about 30 competitive, in four elections.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from congress import war
from senate import stage1, stage2
from senate.regression import lagged

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
EVAL_YEARS = [2018, 2020, 2022, 2024]
IDENTIFIED_YEARS = [2022, 2024]  # windows whose training has more than a handful of races with a lag
WAR_CAP = 10.0  # section 4 only: the lag clipped to +-10 points, a sensitivity check
S1 = stage1.SENATE_S1

data = sr.load_data()
war_races = war.load_war()
lags = war.chamber_lags("Senate")  # one row per race 2006-2026
data_war = war.with_war_lag(data, lags, sr.KEY)  # + war_lag (0 where missing), war_lag_available
assert len(data_war) == len(data) and data_war[sr.KEY].equals(data[sr.KEY])

## 1. The WAR data

**Why center.** Senate WAR is not mean zero within a cycle; House WAR is. A raw lag would carry a national D+2 into
the races that have one.

In [2]:
pd.DataFrame({chamber: war_races[war_races["chamber"] == chamber].groupby("year")["war_dem"].mean()
              for chamber in ["House", "Senate"]}).round(2).rename_axis("mean war_dem")

,House,Senate
mean war_dem,,
2014,-0.11,1.92
2016,0.02,1.87
2018,0.23,3.40
2020,-0.15,2.16
2022,0.15,2.04
2024,0.20,1.25


**Signs.** D positive: Collins (2020) and Manchin (2018) ran far ahead of their states, Tester (2018) too; Gross
(2020) and Kunce (2024) did better than Democrats usually do in Alaska and Missouri.

In [3]:
senate_war = war_races[war_races["chamber"] == "Senate"]
examples = [(2020, "Maine"), (2018, "West Virginia"), (2018, "Montana"), (2020, "Alaska"), (2024, "Missouri")]
senate_war.set_index(["year", "geography"]).loc[examples, ["war_democrat", "war_republican", "war_label", "war_dem",
                                                           "dem_usable", "gop_usable"]]

war_democrat     war_republican war_label  war_dem  dem_usable  gop_usable
year geography                                                                                  
2020 Maine             Sara Gideon      Susan Collins    R+12.5    -12.5        True        True
2018 West Virginia     Joe Manchin  Patrick Morrissey    D+27.3     27.3        True        True
     Montana            Jon Tester     Matt Rosendale    D+14.7     14.7        True        True
2020 Alaska         Al Gross (Ind)       Dan Sullivan     D+1.5      1.5        True        True
2024 Missouri          Lucas Kunce        Josh Hawley     D+7.5      7.5        True        True

**Coverage of the lag**, every race, and whose earlier races it comes from.

In [4]:
flags = lags.assign(flag=lags["war_flag"].str.replace(r" \(.*\)", "", regex=True))
display(pd.crosstab(flags["year"], flags["flag"]).loc[2014:])
display(pd.crosstab(flags["year"], flags["war_from"].fillna("none")).loc[2016:])
display(Markdown("**Not usable** (WAR's anti-Republican candidate wasn't a Democrat, or an off-cycle special): "
                 "these races never enter a history"))
senate_war.loc[~senate_war["dem_usable"] | ~senate_war["gop_usable"],
               ["year", "geography", "war_democrat", "war_republican", "dem_slot", "dem_usable", "gop_usable"]]

flag,neutral: no earlier WAR race of either nominee,one year,two years
year,,,
2014,36,0,0
2016,31,3,0
2018,35,0,0
2020,6,28,1
2022,9,22,4
2024,11,21,2
2026,13,9,13


war_from,both,dem,gop,none
year,,,,
2016,0,1,2,31
2018,0,0,0,35
2020,2,9,18,6
2022,1,11,14,9
2024,3,12,8,11
2026,2,10,10,13


**Not usable** (WAR's anti-Republican candidate wasn't a Democrat, or an off-cycle special): these races never enter a history

,year,geography,war_democrat,war_republican,dem_slot,dem_usable,gop_usable
2335,2014,Kansas,Greg Orman (Ind),Pat Roberts,not_dem_nominee,False,False
2393,2018,Alabama,Doug Jones,Roy Moore,off_cycle,False,False
2429,2020,Arkansas,Ricky Dale Harrington (Libertarian),Tom Cotton,not_dem_nominee,False,False
2440,2020,Louisiana,Various Democrats,Bill Cassidy,unmatched,False,True
2448,2020,Nebraska,Various Democrats,Ben Sasse,unmatched,False,True
2489,2022,Utah,Evan McMullin (Ind),Mike Lee,not_dem_nominee,False,False
2509,2024,Nebraska,Dan Osborn (Ind),Deb Fischer,not_dem_nominee,False,False


## 2. Fits

`n_train_with_L` is the training races with a real lag; the rest carry the neutral 0.

In [5]:
s1_pred = pd.concat([r.predictions for r in stage1.rolling_predictions(S1, eval_years=EVAL_YEARS, data=data).values()],
                    ignore_index=True)
war_pred, war_coef = lagged.fit_predict_lagged(
    data=data_war,
    s1_formula=S1.formula,
    fixed=dict(S1.fixed),
    s1_predictions=s1_pred,
    eval_years=EVAL_YEARS,
    missing_lag="neutral",
    lag=war.LAG,
    available=war.AVAILABLE,
)
# Neutral missing WAR: the same training rows as production S1 in every window, and the same predicted races
for year in EVAL_YEARS:
    s1_rows = int((data["observed"] & data["year"].isin(stage1.expanding_window(S1, year))).sum())
    assert war_coef.loc[year, "n_train"] == s1_rows, year
assert war_pred[sr.KEY].equals(s1_pred[sr.KEY])
PREDICTIONS = {"S1": s1_pred, "S1 + WAR": war_pred}
war_coef

,Intercept,partisan_lean,gen_margin_raw,war_lag,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,,
2018,1.636762,0.991547,0.844903,-0.273283,2.5,198,3,2006-2016
2020,1.614492,0.957614,0.873455,-0.398606,2.5,232,3,2006-2018
2022,1.476269,0.972445,0.849403,0.374971,2.5,266,31,2006-2020
2024,1.706036,0.971605,0.827613,0.275957,2.5,299,56,2006-2022


**Leave one training year out**, on the 2026 window (2006–2024). Only 2016–2024 carry lags.

In [6]:
s1_2026 = stage1.forecast(S1, data=data).predictions
loyo = {}
for dropped in [None, 2016, 2020, 2022, 2024]:
    rows = data_war if dropped is None else data_war[data_war["year"] != dropped]
    _, coef = lagged.fit_predict_lagged(data=rows, s1_formula=S1.formula, fixed=dict(S1.fixed),
                                        s1_predictions=s1_2026, eval_years=[2026], missing_lag="neutral",
                                        lag=war.LAG, available=war.AVAILABLE)
    loyo["none" if dropped is None else dropped] = coef.loc[2026]
pd.DataFrame(loyo).T.rename_axis("dropped")

,Intercept,partisan_lean,gen_margin_raw,war_lag,inc_signed,n_train,n_train_with_L,train_years
dropped,,,,,,,,
none,1.537721,0.965119,0.836294,0.327965,2.5,332,78,2006-2024
2016,1.953342,0.94404,0.821021,0.323892,2.5,300,75,2006-2024
2020,1.638842,0.954333,0.865371,0.274824,2.5,298,50,2006-2024
2022,1.319719,0.966802,0.858193,0.425454,2.5,299,53,2006-2024
2024,1.706036,0.971605,0.827613,0.275957,2.5,299,56,2006-2022


## 3. Out-of-sample comparison, 2018–2024

Fundamentals (Stage 1) and the blend, on the same 134 scored races. `bias` = actual − prediction (negative: too
Democratic). Ex-ante competitive: |lean + eve generic ballot| ≤ 10.

In [7]:
polls, ratings = stage2.load_inputs()
blends = {name: stage2.blend(p, polls, ratings) for name, p in PREDICTIONS.items()}
oos = blends["S1"][[*sr.KEY, "actual", "polling_available"]].merge(
    data_war[[*sr.KEY, "partisan_lean", "gb_margin_2p_norm", "inc_signed", war.LAG, war.AVAILABLE, "war_from"]],
    on=sr.KEY, validate="one_to_one")
for name, b in blends.items():
    assert b[sr.KEY].equals(oos[sr.KEY])
    oos[f"fundamentals: {name}"] = b["margin_fundamentals"].to_numpy()
    oos[f"blend: {name}"] = b["prediction"].to_numpy()
MODELS = {c: c for c in oos.columns if c.startswith(("fundamentals:", "blend:"))}

ex_ante = (oos["partisan_lean"] + oos["gb_margin_2p_norm"]).abs() <= sr.EX_ANTE_THRESHOLD
incumbent, has_lag, polled = oos["inc_signed"] != 0, oos[war.AVAILABLE], oos["polling_available"]
identified = oos["year"].isin(IDENTIFIED_YEARS)
SUBSETS = {
    "all": pd.Series(True, index=oos.index),
    "ex-ante competitive": ex_ante,
    "incumbent": incumbent,
    "open": ~incumbent,
    "ex-ante competitive incumbent": ex_ante & incumbent,
    "with a lag": has_lag,
    "polled": polled,
    "unpolled": ~polled,
    "2022-2024": identified,
    "2022-2024 ex-ante competitive": identified & ex_ante,
    "2022-2024 with a lag": identified & has_lag,
}
pooled = war.score(oos, MODELS, SUBSETS)
pooled.unstack("model").reindex(SUBSETS).round(2)

n                                                               mae                                                              bias                                   \
model                         blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR blend: S1 blend: S1 + WAR fundamentals: S1   
subset                                                                                                                                                                                                         
all                                 134             134              134                    134      4.21            4.28             5.26                   5.68     -1.33           -1.35            -0.15   
ex-ante competitive                  39              39               39                     39      3.13            3.07             3.97                   3.95     -0.60           -0.64             1.17   
incumbent                           105             105              105                    105      4.27            4.36             5.17                   5.70     -1.34           -1.36             0.08   
open                                 29              29               29                     29      3.99            4.00             5.60                   5.62     -1.32           -1.31            -1.00   
ex-ante competitive incumbent        31              31               31                     31      3.41            3.34             4.34                   4.31     -1.00           -1.06             0.86   
with a lag                           75              75               75                     75      4.45            4.57             4.67                   5.41     -1.93           -1.96            -0.65   
polled                               87              87               87                     87      4.00            4.00             5.58                   5.91     -1.55           -1.59            -0.02   
unpolled                             47              47               47                     47      4.61            4.79             4.67                   5.25     -0.93           -0.91            -0.40   
2022-2024                            66              66               66                     66      4.17            4.00             4.83                   4.76     -0.70           -0.71             0.21   
2022-2024 ex-ante competitive        20              20               20                     20      3.29            3.15             4.08                   3.58      0.21            0.18             2.05   
2022-2024 with a lag                 47              47               47                     47      4.00            3.75             4.45                   4.34     -0.54           -0.55             0.51   

                                                     winners_called_pct                                                          
model                         fundamentals: S1 + WAR          blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR  
subset                                                                                                                           
all                                            -0.21              93.28           92.54            93.28                  93.28  
ex-ante competitive                             1.07              84.62           82.05            84.62                  84.62  
incumbent                                       0.01              92.38           91.43            93.33                  93.33  
open                                           -0.99              96.55           96.55            93.10                  93.10  
ex-ante competitive incumbent                   0.74              83.87           80.65            87.10                  87.10  
with a lag                                     -0.75              93.33           92.

### By election year

In [8]:
by_year = pd.concat({year: war.score(oos, MODELS, {"all": oos["year"] == year,
                                                   "ex-ante competitive": (oos["year"] == year) & ex_ante})
                     for year in EVAL_YEARS}, names=["year"])
by_year[["n", "mae", "bias", "winners_called_pct"]].unstack("model").round(2)

n                                                               mae                                                              bias                                   \
model                    blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR blend: S1 blend: S1 + WAR fundamentals: S1   
year subset                                                                                                                                                                                               
2018 all                        34              34               34                     34      3.38            3.38             6.39                   6.39      0.44            0.44             1.74   
     ex-ante competitive         9               9                9                      9      3.01            3.01             4.66                   4.66     -0.17           -0.17             1.92   
2020 all                        34              34               34                     34      5.13            5.73             4.98                   6.76     -4.34           -4.40            -2.73   
     ex-ante competitive        10              10               10                     10      2.93            2.99             3.13                   4.04     -2.60           -2.69            -1.26   
2022 all                        33              33               33                     33      4.28            4.31             5.17                   5.45      0.88            0.91             1.41   
     ex-ante competitive         9               9                9                      9      3.94            3.91             4.67                   4.36      2.29            2.32             2.77   
2024 all                        33              33               33                     33      4.06            3.68             4.49                   4.06     -2.27           -2.32            -1.00   
     ex-ante competitive        11              11               11                     11      2.76            2.52             3.60                   2.95     -1.49           -1.58             1.46   

                                                winners_called_pct                                                          
model                    fundamentals: S1 + WAR          blend: S1 blend: S1 + WAR fundamentals: S1 fundamentals: S1 + WAR  
year subset                                                                                                                 
2018 all                                   1.74              91.18           91.18            91.18                  91.18  
     ex-ante competitive                   1.91              88.89           88.89            88.89                  88.89  
2020 all                                  -2.93              94.12           91.18            97.06                  97.06  
     ex-ante competitive                  -1.52              90.00           80.00           100.00                 100.00  
2022 all                                   1.43              90.91           90.91            90.91                  90.91  
     ex-ante competitive                   2.78              66.67           66.67            66.67                  66.67  
2024 all                                  -1.05              96.97           96.97            93.94                  93.94  
     ex-ante competitive                   1.34              90.91           90.91            81.82                  81.82

### The races that move most

2020–2024, the scored races whose fundamentals prediction moves by more than 3 points.

In [9]:
moved = oos.assign(change=oos["fundamentals: S1 + WAR"] - oos["fundamentals: S1"])
moved = moved[moved["change"].abs() > 3].sort_values("change")
moved[[*sr.KEY, war.LAG, "change", "actual", "fundamentals: S1", "fundamentals: S1 + WAR", "blend: S1",
       "blend: S1 + WAR", "polling_available"]].round(2)

,year,state,class,war_lag,change,actual,fundamentals: S1,fundamentals: S1 + WAR,blend: S1,blend: S1 + WAR,polling_available
87,2022,ND,3,-26.17,-10.03,-38.61,-40.43,-50.47,-37.13,-43.65,False
34,2020,AK,2,16.88,-6.72,-13.36,-12.64,-19.36,-10.85,-13.33,True
77,2022,IA,3,-13.27,-5.02,-12.20,-14.43,-19.45,-11.05,-12.90,True
60,2020,RI,2,12.48,-4.98,33.18,26.57,21.59,28.12,24.88,False
42,2020,ID,2,11.98,-4.76,-30.64,-31.26,-36.03,-31.17,-34.27,False
97,2022,SD,3,-12.17,-4.73,-45.39,-33.10,-37.84,-31.70,-33.46,True
45,2020,KY,2,11.38,-4.53,-20.35,-25.14,-29.66,-16.31,-17.98,True
46,2020,LA,2,9.28,-3.69,-26.84,-15.89,-19.59,-21.18,-23.58,False
108,2024,MA,1,-12.80,-3.46,19.85,32.91,29.45,27.43,26.16,True
130,2024,WA,1,-11.80,-3.21,18.50,19.60,16.40,23.59,21.51,False


## 4. Sensitivity to large WAR values

The same fit with the lag clipped to ±`WAR_CAP` (Collins −21, Warren and Cantwell −12, ...). A check only.

In [10]:
data_capped = war.with_war_lag(data, lags, sr.KEY, cap=WAR_CAP)
capped_pred, capped_coef = lagged.fit_predict_lagged(data=data_capped, s1_formula=S1.formula, fixed=dict(S1.fixed),
                                                     s1_predictions=s1_pred, eval_years=EVAL_YEARS,
                                                     missing_lag="neutral", lag=war.LAG, available=war.AVAILABLE)
capped_blend = stage2.blend(capped_pred, polls, ratings)
assert capped_blend[sr.KEY].equals(oos[sr.KEY])
oos["fundamentals: S1 + WAR (cap)"] = capped_blend["margin_fundamentals"].to_numpy()
oos["blend: S1 + WAR (cap)"] = capped_blend["prediction"].to_numpy()
display(Markdown(f"Scored races with |lag| > {WAR_CAP:g}: **{int((oos[war.LAG].abs() > WAR_CAP).sum())}**. "
                 "Capped coefficient by window: "
                 + ", ".join(f"{y}: {c:.3f}" for y, c in capped_coef[war.LAG].items())))
war.score(oos, {c: c for c in oos.columns if c.startswith(("fundamentals:", "blend:"))},
          {"all": SUBSETS["all"], "2022-2024": identified}).unstack("model")[["mae", "bias"]].round(2)

Scored races with |lag| > 10: **14**. Capped coefficient by window: 2018: -0.273, 2020: -0.399, 2022: 0.472, 2024: 0.399

mae                                                                                                                 bias                                                                                \
model     blend: S1 blend: S1 + WAR blend: S1 + WAR (cap) fundamentals: S1 fundamentals: S1 + WAR fundamentals: S1 + WAR (cap) blend: S1 blend: S1 + WAR blend: S1 + WAR (cap) fundamentals: S1 fundamentals: S1 + WAR   
subset                                                                                                                                                                                                                   
2022-2024      4.17            4.00                   3.9             4.83                   4.76                         4.67     -0.70           -0.71                 -0.75             0.21                   0.19   
all            4.21            4.28                   4.2             5.26                   5.68                         5.51     -1.33           -1.35                 -1.36            -0.15                  -0.21   

                                        
model     fundamentals: S1 + WAR (cap)  
subset                                  
2022-2024                         0.13  
all                              -0.19

## 5. What it would do to 2026 (production unchanged)

S1 and S1 + WAR trained on 2006–2024, each through the production blend with 2026's polls and ratings. NE, ID and SD
have no Democratic nominee; their Stage 2 forecast is centered on the ratings, so their D-vs-R margins here are the
model's, not a contest on the ballot.

In [11]:
war_2026, coef_2026 = lagged.fit_predict_lagged(data=data_war, s1_formula=S1.formula, fixed=dict(S1.fixed),
                                                s1_predictions=s1_2026, eval_years=[2026], missing_lag="neutral",
                                                lag=war.LAG, available=war.AVAILABLE)
blend_2026 = {"S1": stage2.blend(s1_2026, polls, ratings), "S1 + WAR": stage2.blend(war_2026, polls, ratings)}
f26 = blend_2026["S1"][[*sr.KEY, "margin_fundamentals", "prediction", "polling_available"]].merge(
    blend_2026["S1 + WAR"][[*sr.KEY, "margin_fundamentals", "prediction"]], on=sr.KEY, suffixes=(" S1", " S1 + WAR"))
f26 = f26.merge(lags[[*sr.KEY, "dem_name", "gop_name", war.LAG, "war_flag"]], on=sr.KEY, validate="one_to_one")
f26["blend change"] = f26["prediction S1 + WAR"] - f26["prediction S1"]
display(Markdown(f"2026 coefficient: {coef_2026.loc[2026, war.LAG]:.3f} "
                 f"({int(coef_2026.loc[2026, 'n_train_with_L'])} training races with a lag). Blend calls that flip: "
                 f"**{int((np.sign(f26['prediction S1']) != np.sign(f26['prediction S1 + WAR'])).sum())}**."))
f26.sort_values("blend change")[[*sr.KEY, "dem_name", "gop_name", war.LAG, "margin_fundamentals S1",
                                 "margin_fundamentals S1 + WAR", "prediction S1", "prediction S1 + WAR",
                                 "blend change", "polling_available"]].round(2)

2026 coefficient: 0.328 (78 training races with a lag). Blend calls that flip: **0**.

,year,state,class,dem_name,gop_name,war_lag,margin_fundamentals S1,margin_fundamentals S1 + WAR,prediction S1,prediction S1 + WAR,blend change,polling_available
14,2026,ME,2,Troy Jackson,Susan Collins,-20.97,13.60,6.81,8.01,5.50,-2.51,True
13,2026,MA,2,Ed Markey,John Deaton,-10.07,38.12,35.01,35.63,33.61,-2.02,False
33,2026,WV,2,Rachel Fetty Anderson,Shelley Moore Capito,-4.85,-33.98,-35.74,-32.94,-34.08,-1.15,False
23,2026,NM,2,Ben Ray Luján,Larry Marker,-4.86,18.32,16.82,22.76,21.78,-0.98,False
20,2026,NE,2,NaN,Pete Ricketts,-4.05,-13.44,-14.83,-11.71,-12.61,-0.90,False
30,2026,TN,2,Marquita Bradshaw,Bill Hagerty,-3.26,-21.11,-22.27,-24.57,-25.33,-0.76,False
22,2026,NJ,2,Cory Booker,Justin Murphy,-2.42,19.47,18.77,23.50,23.05,-0.45,False
26,2026,OR,2,Jeff Merkley,David Brock Smith,-2.05,25.94,25.40,27.71,27.36,-0.35,False
29,2026,SD,2,NaN,Mike Rounds,-1.25,-21.71,-22.22,-24.96,-25.30,-0.33,False
4,2026,DE,2,Chris Coons,Michael Katz,-1.50,26.64,26.29,28.17,27.94,-0.23,False


## Conclusion

**Not adopted: the evidence doesn't support it.** As specified, S1 + WAR is worse than S1. In the two windows that
can estimate it, it helps, mainly in one election. Production S1 is unchanged.

**1. As specified, 2018–2024 (134 races):**

| | S1 | S1 + WAR |
|---|---|---|
| Fundamentals MAE / bias | **5.26** / −0.15 | 5.68 / −0.21 |
| Fundamentals ex-ante MAE (39) | 3.97 | 3.95 |
| Blend MAE / bias | **4.21** / −1.33 | 4.28 / −1.35 |
| Blend ex-ante MAE | 3.13 | **3.07** |
| Blend winners called | **93.3%** | 92.5% |

The loss is a coverage problem.
- **2018:** no race has a lag (class I senators last ran in 2012).
- **2018 and 2020 windows:** the coefficient comes from the 3 races of 2016 with one, and is negative (−0.27,
  −0.40).
- **2020:** that negative coefficient, applied to real lags, costs 1.8 points of fundamentals MAE. Maine is the
  worst case, moved 16 points the wrong way off Collins' −40 lag.

**2. The identified windows, 2022–2024 (66 races): a modest gain, mostly 2024.**
- **Coefficient:** 0.37 (2022, 31 training races with a lag) and 0.28 (2024, 56). The 2026 window gives 0.33, and
  0.27–0.43 with one year left out. That is below the House's 0.6 and near the residual-based incumbent lag's 0.4–0.5
  (`03_reg_incumbent_lag.ipynb`).
- **MAE, S1 → S1 + WAR:**

  | | Fundamentals | Blend |
  |---|---|---|
  | All 66 races | 4.83 → 4.76 | 4.17 → **4.00** |
  | Ex-ante competitive (20) | 4.08 → 3.58 | 3.29 → 3.15 |

- **Winners called** are unchanged.
- **By year:** 2024 carries it (blend 4.06 → 3.68, fundamentals 4.49 → 4.06); 2022 is level (blend 4.28 → 4.31).
- **Cap:** clipping the lag at ±10 gives a 2022–2024 blend MAE of 3.90.

**3. After the blend most of it is gone.**
- Polled races, where most competitive seats are, are level (4.00 vs 4.00 over 2018–2024): the polls already carry
  candidate strength.
- Unpolled races, mostly safe seats, are worse (4.61 → 4.79).
- This matches the incumbent-lag experiment: persistence is real, but Stage 2's polls and ratings absorb it.

**4. 2026.**
- No blend call flips.
- Maine moves most: Collins' lag (−21) cuts the blend's Jackson margin from 8.0 to 5.5 points.
- Markey loses 2.0 and every other race moves by under 1.2.

**Reading:** one positive test election (2024) and one flat (2022), after polls and ratings, isn't enough to add a
term to the Senate prior. Re-check after 2026: its class II races will add the first full cycle of Senate lags (2020
WAR), and a 2028 window would rest on about 100 training races with a lag (78 now).

**Limitations:**
- **The House notebook's vintage caveat applies:** WAR was recomputed in 2025 with a method chosen after 2024, and
  most of it didn't exist in real time.
- **Senate scope:** only Senate races count, so House WAR of senators who came from the House (Slotkin, Gallego,
  Kim, Peltola in 2026) is left out.
- **Unusable races:** races scored against an independent (Osborn, McMullin, Orman) and the 2017 Alabama special
  have no usable WAR.
- **Missing lag:** a missing lag is a neutral 0 (centered WAR, so the average candidate). Every model trains on S1's
  rows.